# Ego network with alter–alter ties — u/CaseA

Directed reply network built from the [Arctic Shift](https://arctic-shift.photon-reddit.com)
API (Pushshift successor, no credentials required).

**Ego** — u/CaseA.

**Alters** — every account the ego replied to directly, and every account that replied to
the ego directly. "Directly" means a comment whose parent is the other account's content:

- a reply to a **comment** (`parent_id = t1_…`) → the parent comment's author
- a top-level comment on a **submission** (`parent_id = t3_…`) → the submission's author

**Alter–alter ties** — every alter's full archived comment and submission history is pulled,
and any comment by alter *A* whose parent was written by alter *B* becomes a directed
`A → B` tie. Because both endpoints of such a tie are alters, joining each alter's
`parent_id`s against the ids of everything the alters wrote finds every such reply without
resolving millions of parents one by one.

**Edge weight** — the number of **distinct posts** (submissions) on which there is at least
one direct reply in that direction. A comment chain on one post counts once; the raw reply
count is kept as `n_replies`.

**Deleted users are removed.** Comments and submissions whose author is `[deleted]` or
`[removed]` in the archive never become nodes. Every account that Arctic Shift still has a
name for **is kept**, even if the account has since been deleted or suspended. Section 6
looks up each account's *current* status with PRAW (credentials from project 1's
`reddit_env_vars.env`) and stores it as an `account_status` node attribute
(`Active` / `Suspended` / `Deleted`) — it describes nodes, it does not filter them.

**Bots are removed** — accounts on the public Pushshift bot list (`ignored.txt`),
`AutoModerator`, or whose name ends in `bot` (not `robot`). Section 6 prints every removed
account; add a false positive to `NOT_BOTS` to keep it.

**Figures** — (1) ego ↔ alter interactions only; (2) the alter–alter tie network with the
ego removed.

This notebook is separate from `network_case_CaseA_ego_network_v1.ipynb` (star network with a
repetition-discounted weighting) and writes to its own output directory.

---
## API notes verified against the live endpoint

| Finding | Consequence |
|---|---|
| `parent_id` search accepts only `t1_` (comment) parents | replies to the ego's *submissions* come from `link_id` + filter on `parent_id == t3_<id>` |
| `after` is **exclusive** of the timestamp given | paging uses `after = last_created_utc - 1` and dedupes on `id`, so same-second items at a page boundary are not lost |
| `limit='auto'` returns 100–1000 rows | used for history pulls; a page under 100 rows is the last page |
| `fields=` trims the payload | history pulls request only the six columns the join needs |
| `/ids` batches are kept at 100 | an earlier notebook saw silent truncation above that |
| `comments/search?author=` intermittently answers `422` | treated as transient and retried with backoff |
| `users/interactions/users` is capped at 100 rows and mixes reply directions | **not** used — ties are rebuilt from raw comments so each one is traceable |
| `www.reddit.com/user/<name>/about.json` returns `403` anonymously | account status needs PRAW credentials |
| a history page takes ~1 s to serve, far below the 5 req/s cap | histories are pulled by 4 worker threads, each rate-limited to 1.25 req/s (5 req/s combined) |
| u/AutoModerator has ~524M archived comments | it is a bot and removed with the others, so its history is never needed |

## 1. Imports and configuration

In [ ]:
#import libraries
import pandas as pd
import numpy as np
import os
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
import re
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests
from requests_ratelimiter import LimiterSession
from dotenv import load_dotenv
import pyarrow as pa
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
# ---- configuration -------------------------------------------------------
EGO = 'CaseA'          # canonical casing as returned by the API

# Placeholder authors are not accounts and never become nodes.
NON_AUTHORS = {'[deleted]', '[removed]', None, ''}

# Histories too large to pull are skipped and reported rather than silently truncated.
SKIP_HISTORY = set()

# Bots: known list + AutoModerator + a name rule ('...bot', '...bot_2', not '...robot')
BOT_LIST_PATH = '/home/eeporter/redditdumps/PushshiftDumps/scripts/ignored.txt'
EXTRA_BOTS = {'AutoModerator'}
NOT_BOTS = set()                    # names the name rule wrongly flags
BOT_NAME_RULE = re.compile(r'(?<!ro)(bot|b0t)[_-]?\d*$', re.IGNORECASE)
MAX_HISTORY_ITEMS = 250_000

# Columns the alter-alter join needs - requested via `fields=` to keep pages small.
COMMENT_FIELDS = ['id', 'author', 'parent_id', 'link_id', 'created_utc', 'subreddit']
POST_FIELDS    = ['id', 'author', 'created_utc', 'subreddit']

# Parallel history pulls. Combined request rate stays at the project's 5 req/s cap.
N_WORKERS = 4
REQUESTS_PER_SECOND = 5

# PRAW credentials live in project 1's env file (keys: praw_script, praw_key,
# login_name, password, user_agent).
ENV_PATH = '/home/eeporter/Reddit_project1_notebooks/reddit_env_vars.env'

# Derived data stays beside this notebook; figures go to p2_figures.
OUT_DIR   = 'ego_alter_ties_output'
CACHE_DIR = os.path.join(OUT_DIR, 'alter_histories')   # one parquet per alter, resumable
FIG_DIR   = os.path.join('..', 'p2_figures')
for d in (OUT_DIR, CACHE_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

print(f'ego = {EGO}\noutputs -> {OUT_DIR}\nfigures -> {FIG_DIR}')

## 2. API layer

One rate-limited session per thread, each allowed `REQUESTS_PER_SECOND / N_WORKERS`, so the
combined rate never exceeds the cap. Every network loop carries the project's
`retry_count < 10` guard and re-raises once exhausted.

In [ ]:
# Arctic Shift API helpers: one rate-limited session (5 requests/second), a GET with retries, a paginator, and id lookups.
BASE = 'https://arctic-shift.photon-reddit.com/api'
_local = threading.local()


def get_session():
    if not hasattr(_local, 'session'):
        _local.session = LimiterSession(per_minute=int(60 * REQUESTS_PER_SECOND / N_WORKERS))
        _local.session.headers['User-Agent'] = 'dissertation-ego-network/1.0 (research)'
    return _local.session


class BadQuery(Exception):
    """HTTP 400 - a malformed query that retrying cannot fix."""


def arctic_get(path, **params):
    """One GET with the standard retry guard. Raises after 10 failed attempts."""
    retry_count = 0
    last_error = None
    while retry_count < 10:
        try:
            response = get_session().get(f'{BASE}/{path}', params=params, timeout=120)
            if response.status_code == 200:
                return response.json().get('data') or []
            if response.status_code == 400:
                raise BadQuery(f'400 from {path} {params}: {response.text[:200]}')
            last_error = f'HTTP {response.status_code}'
        except BadQuery:
            raise
        except Exception as e:
            last_error = f'{e.__class__.__name__}: {e}'
        retry_count += 1
        print(f'  retry {retry_count} for {path} {params}: {last_error}')
        time.sleep(min(2 ** retry_count, 60))
    raise RuntimeError(f'Failed {path} {params} after 10 retries. Last: {last_error}')


def arctic_paginate(path, limit=100, **params):
    """Page a search endpoint in ascending created_utc order.

    `after` is exclusive, so the cursor is set one second before the last row seen and
    rows are deduplicated on id - nothing posted in the same second as a page boundary
    is skipped.
    """
    rows, seen = [], set()
    after = None
    while True:
        page_params = dict(params, limit=limit, sort='asc')
        if after is not None:
            page_params['after'] = after
        data = arctic_get(path, **page_params)
        new = [r for r in data if r['id'] not in seen]
        for r in new:
            seen.add(r['id'])
        rows.extend(new)
        if len(data) < 100:
            break
        last = data[-1]['created_utc']
        if not new:
            # A full page of already-seen rows: more than a page shares one second.
            # Step past that second rather than loop forever, and say so.
            print(f'  WARNING cursor stalled at {last} for {path} {params}; stepping past it')
            after = last
        else:
            after = last - 1
    return rows


def fetch_by_ids(kind, ids):
    """Look up `posts` or `comments` by id, in batches of 100."""
    ids = [i for i in dict.fromkeys(ids) if i]
    out = []
    for start in range(0, len(ids), 100):
        batch = ids[start:start + 100]
        data = arctic_get(f'{kind}/ids', ids=','.join(batch))
        assert len(data) <= len(batch)
        out.extend(data)
    return out


print('API layer ready')

## 3. The ego's own activity

In [ ]:
# Every comment and post the ego account made.
ego_comments = arctic_paginate('comments/search', author=EGO)
ego_posts    = arctic_paginate('posts/search',    author=EGO)

assert ego_comments or ego_posts, f'no activity found for {EGO} - check the spelling'
authors = {c['author'] for c in ego_comments} | {p['author'] for p in ego_posts}
assert authors <= {EGO}, f'unexpected authors in the ego pull: {authors - {EGO}}'

_ts = [c['created_utc'] for c in ego_comments] + [p['created_utc'] for p in ego_posts]
print(f'ego comments: {len(ego_comments)}')
print(f'ego posts   : {len(ego_posts)}')
print(f'active      : {pd.to_datetime(min(_ts), unit="s")} -> {pd.to_datetime(max(_ts), unit="s")}')

## 4. Ego → alter: who the ego replied to

Parents of the ego's comments are resolved through the `/ids` endpoints. Parents written by
`[deleted]` / `[removed]` are dropped here and counted.

In [ ]:
# Outgoing ties: the author of whatever each ego comment replied to (looked up by id).
# Unresolved parents, deleted authors and self-replies are counted and dropped.
parent_comment_ids = [c['parent_id'][3:] for c in ego_comments if c['parent_id'].startswith('t1_')]
parent_post_ids    = [c['parent_id'][3:] for c in ego_comments if c['parent_id'].startswith('t3_')]

parent_author = {}
parent_author.update({f't1_{c["id"]}': c['author'] for c in fetch_by_ids('comments', parent_comment_ids)})
parent_author.update({f't3_{p["id"]}': p['author'] for p in fetch_by_ids('posts', parent_post_ids)})

ego_interactions = []
dropped = {'unresolved': 0, 'deleted_author': 0, 'self': 0}

for c in ego_comments:
    alter = parent_author.get(c['parent_id'])
    if alter is None:
        dropped['unresolved'] += 1
        continue
    if alter in NON_AUTHORS:
        dropped['deleted_author'] += 1
        continue
    if alter == EGO:
        dropped['self'] += 1
        continue
    ego_interactions.append({
        'source': EGO, 'target': alter,
        'parent_type': 'comment' if c['parent_id'].startswith('t1_') else 'submission',
        'comment_id': c['id'], 'parent_id': c['parent_id'], 'link_id': c['link_id'],
        'created_utc': c['created_utc'], 'subreddit': c['subreddit'],
    })

print(f'ego replies to others : {len(ego_interactions)}')
print(f'dropped               : {dropped}')

## 5. Alter → ego: who replied to the ego

1. Replies to the ego's **comments** — `comments/search?parent_id=t1_<id>`.
2. Top-level comments on the ego's **submissions** — every archived comment on the post via
   `link_id`, kept where `parent_id == t3_<post_id>`.

In [ ]:
# Incoming ties: direct replies to the ego's comments, and top-level comments on the ego's own posts.
dropped_in = {'deleted_author': 0, 'self': 0}

def add_incoming(r):
    if r['author'] in NON_AUTHORS:
        dropped_in['deleted_author'] += 1
        return
    if r['author'] == EGO:
        dropped_in['self'] += 1
        return
    ego_interactions.append({
        'source': r['author'], 'target': EGO,
        'parent_type': 'comment' if r['parent_id'].startswith('t1_') else 'submission',
        'comment_id': r['id'], 'parent_id': r['parent_id'], 'link_id': r['link_id'],
        'created_utc': r['created_utc'], 'subreddit': r['subreddit'],
    })

for c in ego_comments:
    for r in arctic_paginate('comments/search', parent_id=f't1_{c["id"]}'):
        assert r['parent_id'] == f't1_{c["id"]}'
        add_incoming(r)

for p in ego_posts:
    thread = arctic_paginate('comments/search', link_id=f't3_{p["id"]}')
    top_level = [r for r in thread if r['parent_id'] == f't3_{p["id"]}']
    print(f'  post {p["id"]} r/{p["subreddit"]}: {len(thread)} comments, {len(top_level)} top-level')
    for r in top_level:
        add_incoming(r)

ego_df = pd.DataFrame(ego_interactions)
assert ego_df['comment_id'].is_unique, 'one comment recorded twice'

print(f'\nreplies to the ego     : {(ego_df["target"] == EGO).sum()}')
print(f'dropped                : {dropped_in}')

## 6. The alter set, bot removal, and current account status

Archive placeholders (`[deleted]` / `[removed]`) are already gone. Bots are removed next, and
every remaining name is kept. Each account's current status is looked up with PRAW using the same logic as
`account_data()` in `suspicious_id_collection_v1.ipynb`, and recorded — **not** used to
filter. Results are cached so a re-run does not repeat the lookups.

In [ ]:
# Alters = every account in a tie with the ego. Remove bots (known-bot list, name rule, NOT_BOTS exceptions).
alters = sorted((set(ego_df['source']) | set(ego_df['target'])) - {EGO})
assert not set(alters) & NON_AUTHORS

known_bots = {l.strip().lower() for l in open(BOT_LIST_PATH) if l.strip()} | {b.lower() for b in EXTRA_BOTS}
not_bots = {b.lower() for b in NOT_BOTS}


def bot_reason(name):
    n = name.lower()
    if n in not_bots:
        return None
    if n in known_bots:
        return 'known bot'
    if BOT_NAME_RULE.search(name):
        return 'name rule'
    return None


bots = {a: bot_reason(a) for a in alters if bot_reason(a)}
print(f'bots removed: {bots if bots else "none"}')
alters = [a for a in alters if a not in bots]
ego_df = ego_df[~ego_df['source'].isin(bots) & ~ego_df['target'].isin(bots)].reset_index(drop=True)

alter_set = set(alters)
print(f'alters: {len(alters)}')

# Current account status from PRAW for the ego and every alter, cached to parquet; all alters are kept.
assert load_dotenv(ENV_PATH), f'could not load {ENV_PATH}'
missing_keys = [k for k in ('praw_script', 'praw_key', 'user_agent') if not os.getenv(k)]
assert not missing_keys, f'{ENV_PATH} is missing {missing_keys}'

status_path = os.path.join(OUT_DIR, 'account_status.parquet')
account_status = {}
if os.path.exists(status_path):
    account_status = dict(pd.read_parquet(status_path).values)

to_check = [n for n in [EGO] + alters if n not in account_status]
if to_check:
    import praw, prawcore
    reddit_api = praw.Reddit(client_id=os.getenv('praw_script'),
                             client_secret=os.getenv('praw_key'),
                             username=os.getenv('login_name'),
                             password=os.getenv('password'),
                             user_agent=os.getenv('user_agent'),
                             ratelimit_seconds=360)
    for name in to_check:
        retry_count = 0
        while retry_count < 10:
            try:
                redditor = reddit_api.redditor(name)
                try:
                    if hasattr(redditor, 'is_suspended'):
                        account_status[name] = 'Suspended'
                    elif hasattr(redditor, 'id'):
                        account_status[name] = 'Active'
                    else:
                        account_status[name] = 'Deleted'
                except prawcore.exceptions.NotFound:
                    account_status[name] = 'Deleted'
                break
            except prawcore.exceptions.TooManyRequests as e:
                retry_count += 1
                print(f'rate limited on {name}, retry {retry_count}')
                if retry_count == 10:
                    raise e
    (pd.DataFrame(list(account_status.items()), columns=['account', 'account_status'])
       .to_parquet(status_path, engine='pyarrow'))

assert set(account_status) >= {EGO} | alter_set
print(f'ego status: {account_status[EGO]}')
print('alter status (all kept):')
print(pd.Series([account_status[a] for a in alters]).value_counts().to_string())

## 7. Pull every alter's history

Each alter's comments and submissions are cached to `alter_histories/` as soon as the pull
completes, so an interrupted run resumes where it stopped. Expected size is checked first
with `users/search`; accounts in `SKIP_HISTORY` or over `MAX_HISTORY_ITEMS` are not pulled.

**This is the slow step** — roughly 1.4M comments across the alters; expect a couple of
hours on a first run and seconds on a re-run.

In [ ]:
def cache_path(name, kind):
    return os.path.join(CACHE_DIR, f'{name}_{kind}.parquet')

# expected volume per alter (also tells us who is too large to pull)
meta_path = os.path.join(OUT_DIR, 'alter_meta.parquet')
if os.path.exists(meta_path) and set(pd.read_parquet(meta_path)['alter']) >= alter_set:
    alter_meta = pd.read_parquet(meta_path)
else:
    rows = []
    for a in alters:
        hits = [u for u in arctic_get('users/search', author=a) if u['author'].lower() == a.lower()]
        m = hits[0].get('_meta', {}) if hits else {}
        rows.append({'alter': a,
                     'num_comments': m.get('num_comments'),
                     'num_posts': m.get('num_posts')})
    alter_meta = pd.DataFrame(rows)
    alter_meta.to_parquet(meta_path, engine='pyarrow')
alter_meta = alter_meta[alter_meta['alter'].isin(alter_set)].copy()
alter_meta['expected_items'] = alter_meta[['num_comments', 'num_posts']].fillna(0).sum(axis=1)

too_big = set(alter_meta.loc[alter_meta['expected_items'] > MAX_HISTORY_ITEMS, 'alter'])
skipped_history = sorted((too_big | SKIP_HISTORY) & alter_set)
to_pull = [a for a in alters if a not in skipped_history]

print(f'alters to pull           : {len(to_pull)}')
print(f'expected comments + posts: {int(alter_meta.loc[alter_meta["alter"].isin(to_pull), "expected_items"].sum()):,}')
print(f'history NOT pulled       : {skipped_history}')

In [ ]:
def pull_history(a):
    """Pull and cache one alter's comments and posts. Written to a temp file and renamed,
    so an interrupted run never leaves a partial cache file behind."""
    for kind, fields in (('comments', COMMENT_FIELDS), ('posts', POST_FIELDS)):
        path = cache_path(a, kind)
        if os.path.exists(path):
            continue
        rows = arctic_paginate(f'{kind}/search', limit='auto', author=a, fields=','.join(fields))
        df = pd.DataFrame(rows, columns=fields)
        # author search is case-insensitive; keep exactly this account
        df = df[df['author'].str.lower() == a.lower()]
        df.to_parquet(path + '.tmp', engine='pyarrow', compression='zstd')
        os.replace(path + '.tmp', path)
    return a


pending = [a for a in to_pull
           if not (os.path.exists(cache_path(a, 'comments')) and os.path.exists(cache_path(a, 'posts')))]
expected = alter_meta.set_index('alter')['expected_items']
total_items = expected.reindex(pending).sum()
print(f'{len(to_pull) - len(pending)} alters already cached, {len(pending)} to pull '
      f'(~{int(total_items):,} items) with {N_WORKERS} workers')

t_start, done_items = time.time(), 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
    # largest first, so the long pulls run side by side rather than queueing at the end
    futures = [pool.submit(pull_history, a) for a in expected.reindex(pending).sort_values(ascending=False).index]
    for i, fut in enumerate(as_completed(futures), 1):
        a = fut.result()                       # re-raises any collection failure
        done_items += expected[a]
        if i % 10 == 0 or i == len(futures):
            mins = (time.time() - t_start) / 60
            print(f'  {i}/{len(futures)} alters  {done_items / max(total_items, 1):.0%} of items  '
                  f'{mins:.1f} min', flush=True)

In [ ]:
alter_comments = pd.concat(
    [pd.read_parquet(cache_path(a, 'comments')).assign(alter=a) for a in to_pull], ignore_index=True)
alter_posts = pd.concat(
    [pd.read_parquet(cache_path(a, 'posts')).assign(alter=a) for a in to_pull], ignore_index=True)

# pulled vs expected - a large shortfall would mean pagination lost rows
got = (alter_comments.groupby('alter').size().rename('got_comments').to_frame()
       .join(alter_posts.groupby('alter').size().rename('got_posts'), how='outer')
       .reindex(to_pull).fillna(0).astype(int))
coverage = alter_meta.set_index('alter').loc[to_pull, ['num_comments', 'num_posts']].join(got)
coverage['comment_ratio'] = coverage['got_comments'] / coverage['num_comments']

print(f'alter comments: {len(alter_comments):,}   alter posts: {len(alter_posts):,}')
print(f'comments pulled / users-search count: median {coverage["comment_ratio"].median():.3f}, '
      f'min {coverage["comment_ratio"].min():.3f}')
print('lowest coverage:')
coverage.sort_values('comment_ratio').head(5)

## 8. Alter–alter ties

Every piece of content an alter wrote is indexed by its fullname (`t1_<comment id>`,
`t3_<post id>`). An alter comment whose `parent_id` is in that index, and belongs to a
*different* alter, is a direct reply between two alters.

In [ ]:
# Alter-alter ties: an alter's comment that replies to another alter's comment or post (author looked up from the alters' histories).
content_author = pd.concat([
    pd.Series(alter_comments['alter'].values, index='t1_' + alter_comments['id']),
    pd.Series(alter_posts['alter'].values,    index='t3_' + alter_posts['id']),
])
content_author = content_author[~content_author.index.duplicated()]

replies = alter_comments.assign(parent_author=alter_comments['parent_id'].map(content_author))
alter_alter = replies[replies['parent_author'].notna() & (replies['parent_author'] != replies['alter'])]
alter_alter = pd.DataFrame({
    'source': alter_alter['alter'], 'target': alter_alter['parent_author'],
    'parent_type': np.where(alter_alter['parent_id'].str.startswith('t1_'), 'comment', 'submission'),
    'comment_id': alter_alter['id'], 'parent_id': alter_alter['parent_id'],
    'link_id': alter_alter['link_id'], 'created_utc': alter_alter['created_utc'],
    'subreddit': alter_alter['subreddit'],
}).reset_index(drop=True)

print(f'alter -> alter replies: {len(alter_alter):,}')
print(f'distinct directed pairs: {alter_alter[["source", "target"]].drop_duplicates().shape[0]}')
alter_alter.sort_values('created_utc').head(10)

In [ ]:
# --- cross-check: the same join finds alter replies to the ego ----------------
# Anything the join finds must already be in the directly collected alter -> ego set
# (which covers every author, not just alters); a miss would expose a collection gap.
ego_content = {f't1_{c["id"]}' for c in ego_comments} | {f't3_{p["id"]}' for p in ego_posts}
join_to_ego = set(alter_comments.loc[alter_comments['parent_id'].isin(ego_content), 'id'])
direct_to_ego = set(ego_df.loc[ego_df['target'] == EGO, 'comment_id'])
missing = join_to_ego - direct_to_ego
print(f'alter -> ego replies found by the join : {len(join_to_ego)}')
print(f'  of which absent from section 5       : {len(missing)} {sorted(missing)[:10]}')

# --- and the ego -> alter replies: each ego parent should be in the alter index ---
ego_out = ego_df[ego_df['source'] == EGO]
in_index = ego_out['parent_id'].isin(content_author.index)
print(f'ego -> alter replies whose parent is in the alter index: {in_index.sum()}/{len(ego_out)}')
print(f'  parents not found, by alter: {sorted(set(ego_out.loc[~in_index, "target"]))}'
      f'  (history not pulled: {skipped_history})')

## 9. Edge list and graph — undirected

One **undirected** tie per pair of accounts, weighted by the number of **distinct posts**
(`link_id`) the pair replied to each other on, counting a post once however many replies it
carried and whichever way they went. A pair that replied to each other is therefore one tie,
not two.

Direction is kept as columns rather than as edges: `replies_a_to_b` / `replies_b_to_a` and
`reciprocal` (both accounts replied to each other at some point).

In [ ]:
# Undirected ties: one row per unordered pair (ego ties and alter ties), weight = distinct posts with a reply either way.
interactions = pd.concat([ego_df.assign(tie_type='ego'),
                          alter_alter.assign(tie_type='alter')], ignore_index=True)
assert interactions['comment_id'].is_unique, 'a comment counted as two interactions'

pairs = interactions.copy()
a = np.minimum(pairs['source'], pairs['target'])
b = np.maximum(pairs['source'], pairs['target'])
pairs['a'], pairs['b'] = a, b
pairs['a_to_b'] = pairs['source'] == pairs['a']

edges = (pairs
         .groupby(['a', 'b', 'tie_type'])
         .agg(weight=('link_id', 'nunique'),            # distinct posts, either direction
              n_replies=('comment_id', 'size'),
              replies_a_to_b=('a_to_b', 'sum'),
              replies_b_to_a=('a_to_b', lambda s: int((~s).sum())),
              n_comment_parent=('parent_type', lambda s: int((s == 'comment').sum())),
              n_submission_parent=('parent_type', lambda s: int((s == 'submission').sum())),
              first_utc=('created_utc', 'min'),
              last_utc=('created_utc', 'max'),
              subreddits=('subreddit', lambda s: ','.join(sorted(set(s)))))
         .reset_index()
         .rename(columns={'a': 'source', 'b': 'target'}))
edges['reciprocal'] = (edges['replies_a_to_b'] > 0) & (edges['replies_b_to_a'] > 0)
edges = edges.sort_values(['tie_type', 'weight', 'n_replies'],
                          ascending=[True, False, False]).reset_index(drop=True)

# Undirected graph; nodes carry role, account status and whether their history was pulled.
G = nx.Graph()
G.add_node(EGO, role='ego', history_pulled=True, account_status=account_status[EGO])
for al in alters:
    G.add_node(al, role='alter', history_pulled=al not in skipped_history,
               account_status=account_status[al])
for _, e in edges.iterrows():
    G.add_edge(e['source'], e['target'], weight=int(e['weight']), tie_type=e['tie_type'],
               n_replies=int(e['n_replies']), reciprocal=bool(e['reciprocal']))

print(f'nodes: {G.number_of_nodes()}  (ego + {len(alters)} alters)')
print(f'undirected ties: {G.number_of_edges()}')
print(edges.groupby('tie_type').agg(ties=('weight', 'size'), posts=('weight', 'sum'),
                                    replies=('n_replies', 'sum'),
                                    reciprocal=('reciprocal', 'sum')).to_string())
chained = edges[edges['n_replies'] > edges['weight']]
print(f'ties whose replies repeat on the same post: {len(chained)} '
      f'({int((chained["n_replies"] - chained["weight"]).sum())} replies collapsed)')
edges[edges['tie_type'] == 'alter'].head(20)

In [ ]:
# --- checks ----------------------------------------------------------------------
pair_posts = (pairs.groupby(['a', 'b'])['link_id'].nunique()
              .rename_axis(['source', 'target']).rename('posts'))
checks = {
    'graph is undirected': not G.is_directed(),
    'no deleted/removed placeholders as nodes': not set(G.nodes()) & NON_AUTHORS,
    'PRAW-deleted accounts are kept as nodes': all(
        n in G for n, s in account_status.items() if s == 'Deleted'),
    'no self-loops': nx.number_of_selfloops(G) == 0,
    'no bots as nodes': not any(bot_reason(n) for n in G.nodes() if n != EGO),
    'node set is exactly ego + alters': set(G.nodes()) == {EGO} | alter_set,
    'every alter is tied to the ego': all(G.has_edge(EGO, al) for al in alters),
    'one row per unordered pair': not edges.duplicated(subset=['source', 'target']).any(),
    'ties equal the graph edge count': len(edges) == G.number_of_edges(),
    'ego ties carry tie_type ego, others alter': all(
        (d['tie_type'] == 'ego') == (EGO in (u, v)) for u, v, d in G.edges(data=True)),
    'reply counts sum to the interaction count': edges['n_replies'].sum() == len(interactions),
    'weight = distinct posts per unordered pair': (
        edges.set_index(['source', 'target'])['weight'].sort_index() == pair_posts.sort_index()).all(),
    'no weight exceeds its reply count': (edges['weight'] <= edges['n_replies']).all(),
    'direction columns sum to the reply count': (
        edges['replies_a_to_b'] + edges['replies_b_to_a'] == edges['n_replies']).all(),
}
for name, ok in checks.items():
    print(f'  [{"PASS" if ok else "FAIL"}] {name}')
assert all(checks.values())

## 10. Network summary

In [ ]:
# Summary of the alter-only network: ties, density, components, and account status of tied alters.
A = G.subgraph(alters).copy()                  # alters only, ego removed
tied = [n for n in A.nodes() if A.degree(n) > 0]
components = sorted(nx.connected_components(A.subgraph(tied)), key=len, reverse=True)
alter_edges = edges[edges['tie_type'] == 'alter']

print(f'Ego network - u/{EGO}  (undirected, weight = distinct shared posts)')
print('=' * 64)
print(f'alters                          : {len(alters)}')
print(f'ego - alter ties                : {A.number_of_edges() and G.degree(EGO)}')
print(f'  reciprocal (both replied)     : {int(edges.loc[edges.tie_type == "ego", "reciprocal"].sum())}')
print(f'alter - alter ties              : {A.number_of_edges()}  '
      f'({int(alter_edges["weight"].sum())} shared posts, {int(alter_edges["n_replies"].sum())} replies)')
print(f'  reciprocal (both replied)     : {int(alter_edges["reciprocal"].sum())}')
print(f'alters with >=1 alter tie       : {len(tied)} ({len(tied) / len(alters):.1%})')
print(f'alter-only density              : {nx.density(A):.5f}')
print(f'components among tied alters    : {len(components)}  sizes {[len(c) for c in components[:10]]}')
print(f'tie weight range (alter-alter)  : {int(alter_edges["weight"].min())} - '
      f'{int(alter_edges["weight"].max())}')
print(f'history not pulled (ties unknown): {skipped_history}')
print()
print('current account status (PRAW):')
status_table = pd.DataFrame({
    'all alters': pd.Series([account_status[al] for al in alters]).value_counts(),
    'alters with alter ties': pd.Series([account_status[al] for al in tied], dtype=object).value_counts(),
}).fillna(0).astype(int)
print(status_table.to_string())

# One row per alter: alter ties, shared posts with alters and with the ego.
alter_degree = (pd.DataFrame({
    'alter': alters,
    'account_status': [account_status[al] for al in alters],
    'alter_ties': [A.degree(al) for al in alters],
    'alter_shared_posts': [A.degree(al, weight='weight') for al in alters],
    'ego_shared_posts': [G[EGO][al]['weight'] for al in alters],
    'ego_replies': [G[EGO][al]['n_replies'] for al in alters],
    'history_pulled': [al not in skipped_history for al in alters],
}).sort_values(['alter_ties', 'alter_shared_posts'], ascending=False)
  .reset_index(drop=True))
alter_degree.head(20)

## 11. Visualisation

Both figures are undirected: one line per pair, width = distinct shared posts.

**Figure 1 — ego ↔ alter ties.** Ego at the centre, every alter on the ring, node fill =
current account outcome (colours from `Figures/Reddit_figs/BlackPeopleTwitterclusters1.svg`).

**Figure 2 — the alter–alter tie network with the ego**, showing the largest connected group
of alters plus the ego's faint ties to them. Pairs and small groups not connected to that
group are left out of the drawing (their count is in the subtitle) but stay in the saved
graph and tables.

In [ ]:
EGO_COLOR = '#0b0b0b'
TIE_EDGE  = '#7a7873'   # alter - alter tie
EGO_EDGE  = '#b5b3ad'   # ego - alter tie
MUTED     = '#52514e'
SURFACE   = '#fcfcfb'
# account-outcome colours taken from Figures/Reddit_figs/BlackPeopleTwitterclusters1.svg
STATUS_COLORS = {'Active': '#4b2362',      # dark purple
                 'Suspended': '#eeb182',   # tan
                 'Deleted': '#cfccc4',     # neutral grey (not in the source figure)
                 'not checked': '#ffffff'}
status_of = lambda n: account_status.get(n, 'not checked')


def status_handles(names, marker_size=9, lw=0.8):
    counts = pd.Series([status_of(n) for n in names]).value_counts()
    return [Line2D([], [], marker='o', ls='', mfc=STATUS_COLORS[st], mec=MUTED, mew=lw,
                   markersize=marker_size, label=f'{st} ({counts[st]})')
            for st in ('Active', 'Suspended', 'Deleted', 'not checked') if counts.get(st)]


def label_offset(ax, x, y, text, **kw):
    return ax.annotate(text, (x, y), xytext=(0, 6), textcoords='offset points',
                       ha='center', va='bottom', fontsize=7, color='#0b0b0b', zorder=6,
                       bbox=dict(boxstyle='round,pad=0.12', fc=SURFACE, ec='none', alpha=0.75), **kw)


def draw_ego_alter(G, ego, max_labels=30, figsize=(13, 13)):
    alters_ = [n for n in G.nodes() if n != ego]
    w = {n: G[ego][n]['weight'] for n in alters_}
    reps = {n: G[ego][n]['n_replies'] for n in alters_}
    ordered = sorted(alters_, key=lambda n: (-w[n], n.lower()))

    n_alt = len(ordered)
    pos, ang = {ego: np.array([0.0, 0.0])}, {}
    for i, nd in enumerate(ordered):
        ang[nd] = 2 * np.pi * i / n_alt + np.pi / 2
        r = 1.0 if (n_alt <= 60 or i % 2 == 0) else 0.9
        pos[nd] = np.array([r * np.cos(ang[nd]), r * np.sin(ang[nd])])

    wmin, wmax = min(w.values()), max(w.values())
    tied_w = wmax == wmin
    width = lambda x: 0.9 if tied_w else 0.7 + 3.0 * (x - wmin) / (wmax - wmin)

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
    for nd in ordered:
        ax.plot([0, pos[nd][0]], [0, pos[nd][1]], color=EGO_EDGE, lw=width(w[nd]),
                alpha=0.75, solid_capstyle='round', zorder=2)

    size = lambda nd: 34 if tied_w else 34 + 150 * (w[nd] - wmin) / (wmax - wmin)
    ax.scatter([pos[nd][0] for nd in ordered], [pos[nd][1] for nd in ordered],
               s=[size(nd) for nd in ordered],
               c=[STATUS_COLORS[status_of(nd)] for nd in ordered],
               edgecolors=MUTED, linewidths=0.7, zorder=3)
    ax.scatter([0], [0], s=520, c=EGO_COLOR, edgecolors=SURFACE, linewidths=2, zorder=4)
    ax.annotate(f'u/{ego}', (0, 0), xytext=(0, -14), textcoords='offset points', ha='center',
                va='top', fontsize=9, fontweight='bold', color=EGO_COLOR, zorder=6,
                bbox=dict(boxstyle='round,pad=0.2', fc=SURFACE, ec='none', alpha=0.9))

    # heaviest ties first; when every tie weighs the same, space the labels round the ring
    step = max(len(ordered) // max_labels, 1)
    to_label = ordered[::step][:max_labels] if tied_w else ordered[:max_labels]
    for nd in to_label:
        deg_ang = np.degrees(ang[nd])
        flip = 90 < (deg_ang % 360) < 270
        lab = nd if tied_w else f'{nd} ({w[nd]})'
        ax.text(1.04 * np.cos(ang[nd]), 1.04 * np.sin(ang[nd]), lab,
                ha='right' if flip else 'left', va='center', fontsize=7, color='#0b0b0b',
                rotation=deg_ang + 180 if flip else deg_ang, rotation_mode='anchor', zorder=5)

    handles = [Line2D([], [], color=EGO_EDGE, lw=2, label=f'ego - alter tie ({len(ordered)})')]
    handles += status_handles(ordered, lw=0.7)
    ax.legend(handles=handles, loc='upper right', frameon=False, fontsize=8.5,
              title='ties / account outcome', title_fontsize=8.5, alignment='left')
    fig.text(0.02, 0.975, f'Ego-alter ties of u/{ego}', fontsize=13, color='#0b0b0b', va='top')
    fig.text(0.02, 0.948,
             f'{n_alt} alters · undirected · tie weight = distinct shared posts · '
             + ('every pair shares exactly one post, so width is uniform' if tied_w
                else 'width = shared posts')
             + f' · node fill = account outcome · {len(to_label)} alters labelled',
             fontsize=8.5, color=MUTED, va='top')
    ax.set_xlim(-1.45, 1.45); ax.set_ylim(-1.45, 1.45)
    ax.set_aspect('equal'); ax.axis('off')
    fig.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.02)
    return fig, ax


fig, ax = draw_ego_alter(G, EGO)
fig.savefig(os.path.join(FIG_DIR, f'ego_network_{EGO}_ego_alter.png'), dpi=150, facecolor=SURFACE)
plt.show()

In [ ]:
def place_ego(pos, graph, ego, n_grid=31, search=0.3, min_clear=0.06, pass_radius=0.035):
    # Put the ego near the middle of the alter layout, at the closest spot that
    #   1. no alter-alter tie passes through (so no alter tie looks like an ego tie), and
    #   2. leaves room around the node.
    # The alter layout itself is left untouched.
    names = list(pos)
    xy = np.array([pos[n] for n in names])
    centre = xy.mean(axis=0)
    span = np.ptp(xy, axis=0).max()
    P = np.array([pos[u] for u, v in graph.edges()])
    Q = np.array([pos[v] for u, v in graph.edges()])
    D = Q - P
    DD = np.maximum((D ** 2).sum(axis=1), 1e-12)

    best = None
    for gx in np.linspace(-search, search, n_grid):
        for gy in np.linspace(-search, search, n_grid):
            p = centre + span * np.array([gx, gy])
            t = np.clip(((p - P) * D).sum(axis=1) / DD, 0, 1)
            crossings = int((np.linalg.norm(P + t[:, None] * D - p, axis=1) < pass_radius * span).sum())
            clearance = np.linalg.norm(xy - p, axis=1).min() / span
            key = (crossings, clearance < min_clear, np.hypot(gx, gy))
            if best is None or key < best[0]:
                best = (key, p)
    return best[1]


def draw_alter_ties(G, A, tied, ego, max_labels=35, figsize=(13, 13)):
    T = A.subgraph(tied)
    if T.number_of_edges() == 0:
        print('no alter-alter ties to draw')
        return None, None
    comps = sorted(nx.connected_components(T), key=len, reverse=True)
    # sorted node order makes the layout reproducible from run to run
    S = nx.Graph()
    S.add_nodes_from(sorted(comps[0]))
    S.add_edges_from((u, v, d) for u, v, d in T.subgraph(comps[0]).edges(data=True))
    left_out = comps[1:]
    # Kamada-Kawai spreads one connected component evenly
    pos = nx.kamada_kawai_layout(S)
    pos[ego] = place_ego(pos, S, ego)
    deg = dict(S.degree())
    wmax = max(d['weight'] for _, _, d in S.edges(data=True))
    sizes = {n: 70 + 55 * deg[n] for n in S.nodes()}

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)

    # ego ties first and faint, so the alter-alter structure reads on top
    for n in S.nodes():
        ax.plot([pos[ego][0], pos[n][0]], [pos[ego][1], pos[n][1]], color=EGO_EDGE,
                lw=0.8, alpha=0.5, solid_capstyle='round', zorder=1)
    nx.draw_networkx_edges(S, pos, ax=ax, edge_color=TIE_EDGE, alpha=0.9,
                           width=[0.9 + 2.6 * np.log1p(d['weight'] - 1) / max(np.log1p(wmax - 1), 1e-9)
                                  for _, _, d in S.edges(data=True)])
    nx.draw_networkx_nodes(S, pos, ax=ax,
                           node_color=[STATUS_COLORS[status_of(n)] for n in S.nodes()],
                           node_size=[sizes[n] for n in S.nodes()],
                           edgecolors=MUTED, linewidths=0.9)
    ax.scatter([pos[ego][0]], [pos[ego][1]], s=650, c=EGO_COLOR, edgecolors=SURFACE,
               linewidths=2.5, zorder=5)
    alter_labels = [label_offset(ax, *pos[n], n)
                    for n in sorted(S.nodes(), key=lambda n: (-deg[n], n.lower()))[:max_labels]]

    # ego label: try eight positions round the node and keep the one whose rendered box
    # overlaps the fewest alter labels, then the fewest nodes
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    label_boxes = [t.get_window_extent(renderer) for t in alter_labels]
    node_px = ax.transData.transform(np.array([pos[n] for n in S.nodes()]))
    candidates = [((0, -18), 'center', 'top'), ((0, 18), 'center', 'bottom'),
                  ((-20, 0), 'right', 'center'), ((20, 0), 'left', 'center'),
                  ((-16, -16), 'right', 'top'), ((16, -16), 'left', 'top'),
                  ((-16, 16), 'right', 'bottom'), ((16, 16), 'left', 'bottom')]
    best = None
    for offset, ha, va in candidates:
        t = ax.annotate(f'u/{ego}', pos[ego], xytext=offset, textcoords='offset points', ha=ha,
                        va=va, fontsize=9.5, fontweight='bold', color=EGO_COLOR, zorder=8,
                        bbox=dict(boxstyle='round,pad=0.2', fc=SURFACE, ec='none', alpha=0.95))
        bb = t.get_window_extent(renderer).expanded(1.1, 1.3)
        score = (sum(bb.overlaps(b) for b in label_boxes),
                 sum(bb.contains(x, y) for x, y in node_px))
        if best is None or score < best[0]:
            if best is not None:
                best[1].remove()
            best = (score, t)
        else:
            t.remove()

    handles = [Line2D([], [], marker='o', ls='', color=EGO_COLOR, markersize=10, label=f'ego (u/{ego})'),
               Line2D([], [], color=TIE_EDGE, lw=2, label=f'alter - alter tie ({S.number_of_edges()})'),
               Line2D([], [], color=EGO_EDGE, lw=2, label=f'ego - alter tie ({S.number_of_nodes()})')]
    handles += status_handles(S.nodes())
    ax.legend(handles=handles, loc='upper right', frameon=False, fontsize=8.5,
              title='ties / account outcome', title_fontsize=8.5, alignment='left')
    fig.text(0.02, 0.975, f'Ego network of u/{ego} — main connected group of alters',
             fontsize=13, color='#0b0b0b', va='top')
    fig.text(0.02, 0.948,
             f'{S.number_of_nodes()} of {T.number_of_nodes()} tied alters · undirected · '
             f'{len(left_out)} unconnected pair(s)/group(s) of {sum(map(len, left_out))} alters not shown\n'
             f'grey width = distinct shared posts · node size = alter-tie count · '
             f'node fill = account outcome · {min(max_labels, S.number_of_nodes())} most-connected labelled',
             fontsize=8.5, color=MUTED, va='top', linespacing=1.5)
    ax.set_aspect('equal'); ax.axis('off'); ax.margins(0.05)
    fig.subplots_adjust(left=0.02, right=0.98, top=0.93, bottom=0.02)
    return fig, ax


fig, ax = draw_alter_ties(G, A, tied, EGO)
if fig is not None:
    fig.savefig(os.path.join(FIG_DIR, f'ego_network_{EGO}_alters_only.png'), dpi=150, facecolor=SURFACE)
    plt.show()

## 12. Save outputs

In [ ]:
# Save edges, interactions, alter table and graph; check the GraphML reloads as the same undirected graph.
edges.to_parquet(os.path.join(OUT_DIR, f'edges_{EGO}.parquet'), engine='pyarrow', compression='zstd')
interactions.to_parquet(os.path.join(OUT_DIR, f'interactions_{EGO}.parquet'), engine='pyarrow', compression='zstd')
alter_degree.to_parquet(os.path.join(OUT_DIR, f'alters_{EGO}.parquet'), engine='pyarrow', compression='zstd')
nx.write_graphml(G, os.path.join(OUT_DIR, f'ego_network_{EGO}_alter_ties.graphml'))

rb = nx.read_graphml(os.path.join(OUT_DIR, f'ego_network_{EGO}_alter_ties.graphml'))
assert rb.number_of_nodes() == G.number_of_nodes() and rb.number_of_edges() == G.number_of_edges()
assert not rb.is_directed(), 'graph did not round trip as undirected'
assert pd.read_parquet(os.path.join(OUT_DIR, f'edges_{EGO}.parquet')).shape == edges.shape
print('PASS  round trip verified')
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    if os.path.isfile(p):
        print(f'  {p}  ({os.path.getsize(p) / 1024:.1f} KB)')